# [Protected secrets in Kubernetes with CSI](https://developer.hashicorp.com/vault/docs/deploy/kubernetes/vso/csi)

## Set environment variables for Vault and Kubernetes
These variables are used throughout the notebook for configuration.

In [6]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal 
%env K8S_CLUSTER_NAME=cluster.local 

env: WORKDIR=/tmp/vault
env: VAULT_K8S_NAMESPACE=vault
env: VAULT_HELM_RELEASE_NAME=vault
env: VAULT_SERVICE_NAME=vault-internal
env: K8S_CLUSTER_NAME=cluster.local


## Load Vault credentials from environment
This cell loads Vault token, address, and CA cert from the .env file.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')


## Enable CSI in Vault Secrets Operator
This cell upgrades the VSO Helm release to enable CSI support.

In [3]:
%%bash
helm upgrade --namespace vault-secrets-operator --set "csi.enabled=true" vault-secrets-operator hashicorp/vault-secrets-operator

Release "vault-secrets-operator" has been upgraded. Happy Helming!
NAME: vault-secrets-operator
LAST DEPLOYED: Tue Dec  9 07:45:24 2025
NAMESPACE: vault-secrets-operator
STATUS: deployed
REVISION: 2


## Show recent events in VSO namespace
Useful for debugging deployment and CSI issues.

In [8]:
! kubectl get events -n vault-secrets-operator | tail -20

36m         Normal   Pulling             pod/vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4    Pulling image "hashicorp/vault-secrets-operator:1.0.1"
36m         Normal   Pulled              pod/vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4    Successfully pulled image "hashicorp/vault-secrets-operator:1.0.1" in 7.595s (7.595s including waiting). Image size: 83586418 bytes.
36m         Normal   Created             pod/vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4    Created container: manager
36m         Normal   Started             pod/vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4    Started container manager
36m         Normal   SuccessfulCreate    replicaset/vault-secrets-operator-controller-manager-6c8df4c9b4   Created pod: vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4
36m         Normal   ScalingReplicaSet   deployment/vault-secrets-operator-controller-manager              Scaled up replica set vault-secrets-operator-

## List VSO pods
Check the status of Vault Secrets Operator pods.

In [9]:
! kubectl get pods -n vault-secrets-operator

NAME                                                         READY   STATUS    RESTARTS   AGE
vault-secrets-operator-controller-manager-6c8df4c9b4-jtsx4   2/2     Running   0          36m
vault-secrets-operator-csi-node-hwlf8                        3/3     Running   0          3m20s


## Create Vault policy for CSI driver
Defines permissions for reading secrets and AppRole credentials.

In [56]:
%%bash
vault policy write csi-driver-policy - <<EOF
path "example-kv/password" {
    capabilities = ["read"]
}

path "example-kv-v2/data/api-key" {
    capabilities = ["read"]
}

path "sys/license/status" {
    capabilities = ["read"]
}

path "auth/approle/role/my-app/secret-id" {
    capabilities = ["update"]
}

path "auth/approle/role/my-app/role-id" {
    capabilities = ["read"]
}
EOF

Success! Uploaded policy: csi-driver-policy


## Create secrets and AppRole in Vault
Sets up KV secrets and AppRole for CSI access.

In [ ]:
%%bash
# Create Secret and AppRole in Vault
vault secrets enable -path=example-kv kv
vault kv put example-kv/password value="s3cr3tP@ssw0rd"

vault secrets enable -path=example-kv-v2 kv-v2
vault kv put example-kv-v2/api-key value="my-api-key-123456"

vault auth enable approle


Success! Enabled the kv secrets engine at: example-kv/
Success! Data written to: example-kv/password
Success! Enabled the kv-v2 secrets engine at: example-kv-v2/
========= Secret Path =========
example-kv-v2/data/data/api-key

======= Metadata =======
Key                Value
---                -----
created_time       2025-12-09T07:04:47.349999846Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1


Error enabling approle auth: Error making API request.

URL: POST https://127.0.0.1:8443/v1/sys/auth/approle
Code: 400. Errors:

* path is already in use at approle/


Success! Data written to: auth/approle/role/example-role
Key        Value
---        -----
role_id    ee07c673-1a1c-3fe7-2d2f-db6bd1dfdfec
Key                   Value
---                   -----
secret_id             8c085d91-3f1b-2043-3173-ef2019015782
secret_id_accessor    2357f052-f483-33cc-cea5-ade8d2e11f72
secret_id_num_uses    0
secret_id_ttl         0s


## Create AppRole for application
AppRole is used by CSI to authenticate and retrieve secrets.

In [57]:
%%bash
vault write auth/approle/role/my-app token_policies="csi-driver-policy" token_ttl=1h token_max_ttl=4h
vault read auth/approle/role/my-app/role-id
vault write -f auth/approle/role/my-app/secret-id

Success! Data written to: auth/approle/role/my-app
Key        Value
---        -----
role_id    f4d621a4-a9f5-c1fb-89fd-b4d8b35ee355
Key                   Value
---                   -----
secret_id             783a75b5-e24f-13ed-2a9f-18e0e60dd493
secret_id_accessor    2454a60f-bb55-6066-e8d0-f284b7b5e336
secret_id_num_uses    0
secret_id_ttl         0s


## Create Kubernetes role for Vault authentication
Defines how the application service account can authenticate to Vault.

In [78]:
%%bash
vault write auth/kubernetes/role/example-auth-role                        \
    bound_service_account_names=psecret                                   \
    bound_service_account_namespaces=psecret,vault-secrets-operator       \
    token_period=120                                                      \
    token_policies=csi-driver-policy                                      \
    audience=vault


Success! Data written to: auth/kubernetes/role/example-auth-role


## Create namespace and service account for the app
Sets up Kubernetes resources needed for the demo.

In [12]:
%%bash
kubectl create ns psecret
kubectl create sa psecret -n psecret

namespace/psecret created
serviceaccount/psecret created


## Create service account in VSO namespace
Required for Vault authentication and CSI access.

In [25]:
! kubectl create sa psecret -n vault-secrets-operator

serviceaccount/psecret created


## Create VaultAuth and VaultAuthGlobal resources
Configure Vault authentication for the application and CSI driver.

In [80]:
%%bash
cat > ${WORKDIR}/vaultauth2_crd.yaml <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
 name: example2
 namespace: vault-secrets-operator
spec:
 vaultAuthGlobalRef:
   name: default

---

apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuthGlobal
metadata:
 name: default
 namespace: vault-secrets-operator
spec:
 vaultConnectionRef: example
 defaultAuthMethod: kubernetes
 kubernetes:
   audiences:
     - vault
   mount: kubernetes
   role: example-auth-role
   serviceAccount: psecret
   tokenExpirationSeconds: 600

EOF

kubectl apply -f ${WORKDIR}/vaultauth2_crd.yaml

vaultauth.secrets.hashicorp.com/example2 created
vaultauthglobal.secrets.hashicorp.com/default created


## Describe VaultAuth resource
Check status and details of VaultAuth configuration.

In [61]:
! kubectl describe VaultAuth example2 -n vault-secrets-operator

Name:         example2
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultAuth
Metadata:
  Creation Timestamp:  2025-12-09T08:42:49Z
  Generation:          1
  Resource Version:    14463
  UID:                 4b513721-8a31-4e5e-8bb7-582368ab02f1
Spec:
  Vault Auth Global Ref:
    Name:  default
Events:    <none>


## Create CSISecrets resource
Defines which secrets are exposed to the application via CSI.

In [103]:
%%bash
cat > ${WORKDIR}/csisecrets_crd.yaml <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: CSISecrets
metadata:
  name: my-app-secrets
  namespace: vault-secrets-operator
spec:
  vaultAuthRef:
    name: example2
  secrets:
    vaultStaticSecrets:
      - mount: example-kv
        path: password
        type: kv-v1
      - mount: example-kv-v2
        path: api-key
        type: kv-v2
        version: 1 # The version of the KV secret -- if not specified, defaults to the latest version
    vaultAppRoleSecretIDs:
      - role: my-app
        mount: approle
        wrapTTL: "30m"
        ttl: "1h"
        numUses: 2
  accessControl:
    matchPolicy: any
    serviceAccountPattern: "^psecret$"
    namespacePatterns:
      - "^psecret$"
    #podNamePatterns:
    #  - "^my-app-"
  syncConfig:
    containerState:
      namePattern: "^(app|sidecar)$"


EOF
kubectl apply -f ${WORKDIR}/csisecrets_crd.yaml

csisecrets.secrets.hashicorp.com/my-app-secrets created


## Describe CSISecrets resource
Check status and details of the CSISecrets configuration.

In [111]:
! kubectl describe CSISecrets -n vault-secrets-operator my-app-secrets 

Name:         my-app-secrets
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         CSISecrets
Metadata:
  Creation Timestamp:  2025-12-09T09:07:05Z
  Generation:          1
  Resource Version:    18696
  UID:                 92a7dcb0-5c7b-4431-b834-dd9d19b12edd
Spec:
  Access Control:
    Match Policy:  any
    Namespace Patterns:
      ^psecret$
    Service Account Pattern:  ^psecret$
  Secrets:
    Vault App Role Secret I Ds:
      Mount:     approle
      Num Uses:  2
      Role:      my-app
      Ttl:       1h
      Wrap TTL:  30m
    Vault Static Secrets:
      Mount:    example-kv
      Path:     password
      Type:     kv-v1
      Mount:    example-kv-v2
      Path:     api-key
      Type:     kv-v2
      Version:  1
  Sync Config:
    Container State:
      Name Pattern:  ^(app|sidecar)$
  Vault Auth Ref:
    Name:  example2
Events:
  Type    Reason        Age                  From            Me

## Create RBAC for CSI driver
Grants the CSI node service account permission to read CSISecrets resources.

In [106]:
%%bash
# Create ClusterRole to read CSISecrets
cat > ${WORKDIR}/csi-rbac.yaml <<EOF
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: vso-csi-secrets-reader
rules:
- apiGroups: ["secrets.hashicorp.com"]
  resources: ["csisecrets"]
  verbs: ["get", "list", "watch"]

---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: vso-csi-secrets-reader
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: vso-csi-secrets-reader
subjects:
- kind: ServiceAccount
  name: vault-secrets-operator-csi-node
  namespace: vault-secrets-operator

EOF

kubectl apply -f ${WORKDIR}/csi-rbac.yaml

clusterrole.rbac.authorization.k8s.io/vso-csi-secrets-reader created
clusterrolebinding.rbac.authorization.k8s.io/vso-csi-secrets-reader created
clusterrolebinding.rbac.authorization.k8s.io/vso-csi-secrets-reader created


# Create Application and mount secrets

In [133]:
%%bash

cat > ${WORKDIR}/my_app_deployment.yaml <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: my-app
  namespace: psecret
  labels:
    app.kubernetes.io/component: my-app
spec:
  selector:
    matchLabels:
      app.kubernetes.io/component: my-app
  replicas: 2
  template:
    metadata:
      labels:
        app.kubernetes.io/component: my-app
    spec:
      serviceAccountName: psecret
      containers:
      - name: app
        image: nginx:latest
        volumeMounts:
        - name: csi-secrets
          mountPath: /var/run/csi-secrets
      - name: sidecar
        image: redis:latest
        volumeMounts:
        - name: csi-secrets
          mountPath: /var/run/csi-secrets
      volumes:
      - name: csi-secrets
        csi:
          driver: csi.vso.hashicorp.com
          volumeAttributes:
            csiSecretsName: my-app-secrets
            csiSecretsNamespace: vault-secrets-operator

EOF
kubectl apply -f ${WORKDIR}/my_app_deployment.yaml



deployment.apps/my-app created


In [136]:
! kubectl get pods -n psecret

NAME                      READY   STATUS    RESTARTS   AGE
my-app-5975d7f687-n69pr   2/2     Running   0          9s
my-app-5975d7f687-tx4j2   2/2     Running   0          9s


In [ ]:
%%bash
# REad secrets via exec
export POD=$(kubectl get pods -n psecret -l app.kubernetes.io/component=my-app -o jsonpath="{.items[0].metadata.name}")
kubectl exec -i -t $POD -n psecret -- ls /var/run/csi-secrets/

Defaulted container "app" out of: app, sidecar
Unable to use a TTY - input is not a terminal or the right kind of file
right kind of file


app_role_0_wrap_info.json
static_secret_0_value
static_secret_1_value
